# Chapter 08: Quickest Change Detection: CUSUM & Shiryaev-Roberts
### *Page's CUSUM, the Shiryaev-Roberts Procedure, and Lorden's Minimax Delay Theorem*

**[Course Home](../README.md) | [Python Lab](../python/08_quickest_change_detection_cusum_and_shiryaev_roberts.ipynb) | [Prev: Ch 07](07_multi_armed_bandits_gittins_and_whittle_indices.ipynb) | [Next: Ch 09](09_mechanism_design_and_tragedy_of_the_commons.ipynb)**

---

## 1. Motivation: Quickest Change Detection in Production Telemetry

Modern engineering systems operate under continuous telemetry surveillance:
* Monitoring error rates and latency distributions across microservice RPC endpoints.
* Tracking continuous integration (CI/CD) test suite execution times to catch silent performance regressions.
* Detecting intermittent hardware faults or race conditions in worker pools immediately after a new software release is deployed.

In all these scenarios, the system undergoes an unannounced change in distribution at an unknown time $\tau \in \{1, 2, \dots, \infty\}$:
* For $t < \tau$: Telemetry observations follow nominal baseline noise distribution $P_0$.
* For $t \ge \tau$: Telemetry observations follow an anomalous or degraded distribution $P_1$.

### Why Ad-Hoc Moving Windows Fail
The prevailing industry practice relies on **moving average windows** (e.g., "calculate error rate over the last 50 requests; alert if $> 5\%$"):
1. **The Window Sizing Dilemma**:
   * If the window $W$ is small ($W = 10$), sample variance is massive, triggering frequent false alarms that wake on-call engineers for benign noise.
   * If the window $W$ is large ($W = 500$), small persistent shifts are smoothed out, and sudden catastrophic breakages take hundreds of requests to detect, severely violating Service Level Agreements (SLAs).
2. **Loss of Historical Evidence**:
   A moving window rigidly drops observations older than $W$. If a subtle regression causes a shift that requires 60 observations to establish statistical significance with $99.9\%$ confidence, a sliding window of length 50 will *never* detect it!

```
 Sliding Window (Rigid Memory W):
 [ y_{t-W+1}, ..., y_t ]  <-- Drops all prior evidence! High variance or high lag.
 
 CUSUM / Shiryaev-Roberts (Optimal Memory):
 S_t = max(0, S_{t-1} + z_t) <-- Integrates all evidence while resetting benign noise!
```

**Quickest Change Detection (QCD)** provides a mathematically rigorous formulation:
Given a user-specified bound on the false alarm rate (e.g., no more than 1 false alarm per $10{,}000$ hours), what stopping time $T$ provably **minimizes the worst-case detection delay** $(T - \tau)^+$?

---

## 2. Mathematical Foundations of Quickest Change Detection

### 2.1 The Problem Formulation
Let $Y_1, Y_2, \dots, Y_t$ be a sequence of observations taking values in $\mathbb{R}^d$.
There exists an unknown, deterministic or random change-point $\tau \in \{1, 2, \dots, \infty\}$ such that conditioned on $\tau$:
* $Y_1, \dots, Y_{\tau-1} \stackrel{\text{i.i.d.}}{\sim} P_0$ with probability density $p_0(y)$.
* $Y_\tau, Y_{\tau+1}, \dots \stackrel{\text{i.i.d.}}{\sim} P_1$ with probability density $p_1(y)$, where $P_1 \ne P_0$.

We seek an adapted stopping time $T$ with respect to the filtration $\mathcal{F}_t = \sigma(Y_1, \dots, Y_t)$ that balances:
1. **False Alarm Rate**: The **Average Run Length to False Alarm** under the null distribution ($P_\infty$, where $\tau = \infty$):

$$
\text{ARL}_0 = \mathbb{E}_\infty[T] \ge \gamma
$$

where $\gamma$ is a large operational threshold (e.g., $\gamma = 10^4$).
2. **Detection Delay**: The expected time to declare a change after the change occurs.

---

## 3. Page's Cumulative Sum (CUSUM) Procedure

### 3.1 Derivation of CUSUM
Introduced by E. S. Page in 1954, the CUSUM algorithm can be derived as a sequential Generalized Likelihood Ratio (GLR) test for an unknown change-point.
If the change occurred at step $k \le t$, the log-likelihood ratio of the sample $Y_1, \dots, Y_t$ is:

$$
\Lambda_{k, t} = \sum_{i=k}^t \ln\left(\frac{p_1(Y_i)}{p_0(Y_i)}\right) = \sum_{i=k}^t z_i
$$

Since the actual change time $k$ is unknown, we maximize over all possible change points $1 \le k \le t$:

$$
S_t = \max_{1 \le k \le t+1} \sum_{i=k}^t z_i
$$

with the convention that $\sum_{i=t+1}^t z_i = 0$.
Crucially, this maximum satisfies a recursive Bellman-style update:

$$
S_0 = 0
$$

$$
S_t = \max\left(0, \; S_{t-1} + z_t\right), \quad z_t = \ln\left(\frac{p_1(Y_t)}{p_0(Y_t)}\right)
$$

Stopping Rule:

$$
T_{\text{CUSUM}} = \inf \{ t \ge 1 : S_t \ge h \}
$$

where $h > 0$ is the decision threshold calibrated to satisfy $\text{ARL}_0 \ge \gamma$.

```
 CUSUM Statistic S_t
            ^
            |       ANOMALY DETECTED! (Alarm Triggered)
   Threshold h +==============================================
            |                            /\
            |                           /  \
            |                          /    \
            |        /\               /
            |  /\   /  \             /
   Floor 0  +--+---+----+-----------+-------------------------> Time t
            |  Reset     Reset      |
            |  to 0      to 0       Change Point \tau
```

The operation $\max(0, \cdot)$ provides the fundamental renewal mechanism:
* When telemetry is nominal ($t < \tau$), $\mathbb{E}_0[z_t] = -D_{\text{KL}}(P_0 \parallel P_1) < 0$. The score drifts downward with negative velocity, constantly hitting the reflective barrier at $0$.
* When a change occurs ($t \ge \tau$), $\mathbb{E}_1[z_t] = D_{\text{KL}}(P_1 \parallel P_0) > 0$. The score drifts upward with positive velocity, crossing $h$ in minimum expected time.

---

## 4. The Shiryaev-Roberts (SR) Procedure

### 4.1 Derivation via Quasi-Bayesian Integration
Whereas CUSUM takes the supremum over unknown change points ($S_t = \max_{k} \Lambda_{k, t}$), the **Shiryaev-Roberts** procedure computes a sum over all possible change points under a uniform improper prior $\mathbb{P}(\tau = k) \propto 1$:

$$
R_t = \sum_{k=1}^t \prod_{i=k}^t \frac{p_1(Y_i)}{p_0(Y_i)}
$$

This sum satisfies the linear recurrence:

$$
R_0 = 0
$$

$$
R_t = (1 + R_{t-1}) \frac{p_1(Y_t)}{p_0(Y_t)}
$$

Stopping Rule:

$$
T_{\text{SR}} = \inf \{ t \ge 1 : R_t \ge A \}
$$

### 4.2 The Shiryaev-Roberts Martingale Property
The SR procedure possesses an exact mathematical property: Under $P_0$ (no change occurs), the process $M_t = R_t - t$ is a zero-mean $\mathcal{F}_t$-martingale!
By Doob's Optional Stopping Theorem:

$$
\mathbb{E}_0[M_{T_{\text{SR}}}] = \mathbb{E}_0[R_{T_{\text{SR}}} - T_{\text{SR}}] = 0 \implies \mathbb{E}_0[T_{\text{SR}}] = \mathbb{E}_0[R_{T_{\text{SR}}}] \ge A
$$

This provides an exact, non-asymptotic lower bound on the false alarm run length:

$$
\text{ARL}_0 = \mathbb{E}_\infty[T_{\text{SR}}] \ge A
$$

Setting the threshold $A = \gamma$ guarantees that the Average Run Length to false alarm is at least $\gamma$ with zero calibration guesswork.

---

## 5. Lorden's Minimax Criterion & Delay Theorem

### 5.1 Lorden's Minimax Delay Metric
In 1971, Gary Lorden formalized the gold-standard optimization criterion for quickest change detection. To guard against adversarial change points and worst-case sample histories prior to change, Lorden defined the **worst-case average detection delay**:

$$
\bar{\mathbb{E}}_1[T - \tau] = \sup_{\tau \ge 1} \text{ess sup} \, \mathbb{E}_\tau \left[ (T - \tau + 1)^+ \mid \mathcal{F}_{\tau-1} \right]
$$

Lorden's Minimax Problem:

$$
\min_{T} \bar{\mathbb{E}}_1[T - \tau] \quad \text{subject to} \quad \mathbb{E}_\infty[T] \ge \gamma
$$

### 5.2 Lorden's Asymptotic Theorem
Lorden proved that as the false alarm constraint $\gamma \to \infty$, the minimum possible detection delay across *any* causally admissible stopping time is governed strictly by the Kullback-Leibler divergence between the post-change and pre-change distributions:

$$
\bar{\mathbb{E}}_1[T - \tau] \ge \frac{\ln \gamma}{D_{\text{KL}}(P_1 \parallel P_0)} \quad \text{as } \gamma \to \infty
$$

Furthermore, Lorden proved that Page's CUSUM procedure with threshold $h = \ln \gamma$ asymptotically achieves this lower bound:

$$
\bar{\mathbb{E}}_1[T_{\text{CUSUM}} - \tau] = \frac{\ln \gamma}{D_{\text{KL}}(P_1 \parallel P_0)} + O(1)
$$

### 5.3 Moustakides' Exact Optimality Theorem
In 1986, George Moustakides proved the ultimate result for quickest change detection:
Page's CUSUM is not merely asymptotically optimal—**CUSUM is strictly optimal for all thresholds $h > 0$**! For any value of $\text{ARL}_0 \ge \gamma$, CUSUM uniquely minimizes Lorden's worst-case detection delay among all possible sequential procedures.

---

## 6. Concrete Step-by-Step Worked Examples with Real Numbers

### 6.1 Worked Example 1: Microservice Error Rate Shift
An API gateway monitors failure codes on an authentication microservice:
* Nominal baseline error rate: $p_0 = 0.01$ ($1\%$ baseline transient network errors).
* Post-change degraded error rate: $p_1 = 0.06$ ($6\%$ failure rate due to a degraded cache instance).
* SLA requirement: No more than 1 false alarm per $20{,}000$ incoming requests ($\text{ARL}_0 \ge 20{,}000$).

#### Step 1: Compute Kullback-Leibler Divergences
The KL divergence between post-change and pre-change Bernoulli distributions:

$$
D_{\text{KL}}(P_1 \parallel P_0) = p_1 \ln\left(\frac{p_1}{p_0}\right) + (1 - p_1) \ln\left(\frac{1 - p_1}{1 - p_0}\right)
$$

Substitute the real numbers:

$$
\frac{p_1}{p_0} = \frac{0.06}{0.01} = 6.0 \implies \ln(6.0) \approx 1.79176 \text{ nats}
$$

$$
\frac{1 - p_1}{1 - p_0} = \frac{0.94}{0.99} \approx 0.949495 \implies \ln(0.949495) \approx -0.05184 \text{ nats}
$$

$$
D_{\text{KL}}(P_1 \parallel P_0) = 0.06(1.79176) + 0.94(-0.05184) = 0.10751 - 0.04873 = 0.05878 \text{ nats/request}
$$

The reverse KL divergence (drift velocity under nominal noise):

$$
D_{\text{KL}}(P_0 \parallel P_1) = 0.01 \ln\left(\frac{0.01}{0.06}\right) + 0.99 \ln\left(\frac{0.99}{0.94}\right) = 0.01(-1.79176) + 0.99(+0.05184) = -0.01792 + 0.05132 = 0.03340 \text{ nats/request}
$$

#### Step 2: Calibrate the CUSUM Threshold
To guarantee $\text{ARL}_0 \ge 20{,}000$, we set:

$$
h = \ln(\text{ARL}_0) = \ln(20000) \approx 9.90349 \text{ nats}
$$

#### Step 3: Compute Expected Lorden Detection Delay
Using Lorden's Theorem:

$$
\mathbb{E}[\text{Delay}] \approx \frac{h}{D_{\text{KL}}(P_1 \parallel P_0)} = \frac{9.90349}{0.05878} \approx 168.49 \text{ requests}
$$

At an ingestion rate of 10 requests per second, the CUSUM detector will flag the cache regression in **$16.8$ seconds**, while guaranteeing that a false alarm occurs on average only once every $2{,}000$ seconds of continuous operation!

#### Step 4: Step-by-Step CUSUM Trajectory Trace
Let us trace the score $S_t$ over a sample transition where change occurs at $t = 5$:
* Log-likelihood increments:
  * On clean response ($Y_t = 0$): $z = -0.05184$ nats.
  * On error response ($Y_t = 1$): $z = +1.79176$ nats.
* Observations:
  * $t = 1 (Y=0): S_1 = \max(0, 0 - 0.05184) = 0$ (Reset at 0)
  * $t = 2 (Y=0): S_2 = \max(0, 0 - 0.05184) = 0$ (Reset at 0)
  * $t = 3 (Y=1 \text{ noise}): S_3 = \max(0, 0 + 1.79176) = 1.79176$
  * $t = 4 (Y=0): S_4 = \max(0, 1.79176 - 0.05184) = 1.73992$
  * $t = 5 (\text{Change occurs: } Y=1): S_5 = 1.73992 + 1.79176 = 3.53168$
  * $t = 6 (Y=1): S_6 = 3.53168 + 1.79176 = 5.32344$
  * $t = 7 (Y=0): S_7 = 5.32344 - 0.05184 = 5.27160$
  * $t = 8 (Y=1): S_8 = 5.27160 + 1.79176 = 7.06336$
  * $t = 9 (Y=1): S_9 = 7.06336 + 1.79176 = 8.85512$
  * $t = 10 (Y=1): S_{10} = 8.85512 + 1.79176 = 10.64688 \ge h = 9.9035$

Alarm triggers at $t = 10$ ($5$ steps after the change)!

---

## 7. Engineering Takeaways and Practical Architecture

| Metric | Moving Average Window | Page's CUSUM | Shiryaev-Roberts (SR) |
| :--- | :--- | :--- | :--- |
| **Optimality** | None (ad-hoc heuristic) | Strictly Minimax Optimal (Moustakides 1986) | Optimal for expected delay under uniform prior |
| **Threshold Calibration** | Empirical tuning of window size $W$ and cutpoint | Analytic: $h \approx \ln(\text{ARL}_0)$ | Exact: $A = \text{ARL}_0$ via Martingale theorem |
| **Memory Requirement** | $O(W)$ buffer storage | $O(1)$ scalar accumulator | $O(1)$ scalar accumulator |
| **Detection Lag** | Substantial ($W/2$ delay) | Minimized: $\frac{\ln \gamma}{D_{\text{KL}}(P_1 \parallel P_0)}$ | Minimized: $\frac{\ln A}{D_{\text{KL}}(P_1 \parallel P_0)}$ |
| **Reset Dynamics** | Rigid truncation at $t - W$ | Dynamic barrier reset at 0 | Multiplicative weight accumulation |

---
**[Course Home](../README.md) | [Python Lab](../python/08_quickest_change_detection_cusum_and_shiryaev_roberts.ipynb) | [Prev: Ch 07](07_multi_armed_bandits_gittins_and_whittle_indices.ipynb) | [Next: Ch 09](09_mechanism_design_and_tragedy_of_the_commons.ipynb)**
